This Kaggle notebook serves as an undergraduate final project for a Spring 2026 course on Natural Language Processing & Computational Linguistics at the University of Alaska Fairbanks. By Ian Rodriguez. The purpose of this project is to fine-tune ScandiBERT to return independent probabilities that a given text is Norwegian, Swedish, Danish, or Non-Scandinavian / None of the Above; this project is "neuro-symbolic", embracing restraints in training data while using regex to enforce linguistic heuristics to make up for it.

 
Original Model: [ScandiBERT by Snæbjarnarson, et al.](https://huggingface.co/vesteinn/ScandiBERT)

Training/Validation Data: [OPUS-100 by Helsinki NLP / Zhang, et al.](https://huggingface.co/datasets/Helsinki-NLP/opus-100)

Reference & Final Validation Dataset for Comparison: [SLIDE by Fedorova, et al.](https://huggingface.co/datasets/ltg/slide)

[This project is licensed under AGPL-3.0.](https://www.gnu.org/licenses/agpl-3.0.en.html)

([GitHub](https://github.com/cloudeerie/scandiprob) | [Hugging Face](https://huggingface.co/ianro04/ScandiProb))

In [1]:
import os
import warnings
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install transformers
!pip install sentencepiece
!pip install datasets

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("kaggle-scandibert")

import torch
import math
import re
from huggingface_hub import hf_hub_download
from transformers import XLMRobertaTokenizer, AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset, concatenate_datasets
from huggingface_hub import snapshot_download

model_name = "vesteinn/ScandiBERT"
labels = ["Norwegian", "Swedish", "Danish", "Non-Scandinavian"]
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in enumerate(labels)}

print("Downloading model components (ignoring broken tokenizer.json)...")
model_path = snapshot_download(
    repo_id="vesteinn/ScandiBERT", 
    ignore_patterns=["tokenizer.json"]
)

tokenizer = XLMRobertaTokenizer.from_pretrained(model_path)

SENTENCES_PER_CLASS = 6000 # +24 handwritten multi-class/lang sentences across all_scandi_sent and da_no_cross_sent
assert 2000 <= SENTENCES_PER_CLASS <= 20000, "SENTENCES_PER_CLASS is outside reasonable range!"

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = len(labels),
    id2label = id2label,  
    label2id = label2id,
    ignore_mismatched_sizes = True,
    problem_type = "multi_label_classification"
)

def get_opus_sentences(config, lang_code, num_sentences=SENTENCES_PER_CLASS, split_type="train"):
    assert (split_type.strip().lower() in ["train", "validation", "test"]), "Invalid dataset type for split (OPUS)."
    print(f"Downloading {lang_code} from {config}...")
    ds = load_dataset("Helsinki-NLP/opus-100", config, split=split_type)
    if split_type == "train":
        ds = ds.filter(lambda sentence: is_dataset_clean(sentence["translation"][lang_code]))
    ds = ds.shuffle(seed=42).select(range(num_sentences))
    return [item[lang_code] for item in ds["translation"]]

def is_dataset_clean(text, instance_threshold=2, min_words=1, min_chars=6):
    bad_char_regex = r"[@#$%^&*_=+\[\]{};':\"<>/`~\\|]"
    bad_format_regex = r'[A-Z][a-z]+[A-Z]|%\s*\d|https?://'
    if not isinstance(text, str):
        return False
    if len(text.strip()) < min_chars or len(text.split()) < min_words:
        return False
    if len(re.findall(bad_char_regex, text)) >= instance_threshold:
        return False
    if re.search(bad_format_regex, text):
        return False
    return True        

def sent_based_int(percent, sent_const=SENTENCES_PER_CLASS): # Get an integer that's a % of SENTENCES_PER_CLASS
    return int(math.ceil(sent_const * percent))

sample_norwegian_all = get_opus_sentences("en-nb", "nb", sent_based_int(0.75)) + get_opus_sentences("en-nn", "nn", sent_based_int(0.25))
sample_swedish_all = get_opus_sentences("en-sv", "sv")
sample_danish_all = get_opus_sentences("da-en", "da")
nonscandi_configs = [("en-es", "en"), ("en-es", "es"), ("en-is", "is"), ("de-en", "de"), ("en-nl", "nl"), ("af-en", "af")]
sample_nonscandi_all = []
for config, code in nonscandi_configs:
    sample_nonscandi_all.extend(get_opus_sentences(config, code, sent_based_int(1 / len(nonscandi_configs))))
print(f"Norwegian corpora collected: {len(sample_norwegian_all)} sentences.")
print(f"Swedish corpora collected: {len(sample_swedish_all)} sentences.")
print(f"Danish corpora collected: {len(sample_danish_all)} sentences.")
print(f"Non-Scandinavian corpora collected: {len(sample_nonscandi_all)} sentences, with {len(sample_nonscandi_all) / len(nonscandi_configs)} sentences per language.")

test_text = "Dette er en norsk setning."
inputs = tokenizer(test_text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
print("\nZero-Shot Classification Probabilities: ")
probs = torch.sigmoid(outputs.logits)
for label, prob in zip(labels, probs[0]): # True "zero-shot", random classification test
    print(f"{label}: {prob.item():.4f}")

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: vesteinn/ScandiBERT
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


README.md: 0.00B [00:00, ?B/s]

en-nb/test-00000-of-00001.parquet:   0%|          | 0.00/223k [00:00<?, ?B/s]

en-nb/train-00000-of-00001.parquet:   0%|          | 0.00/10.2M [00:00<?, ?B/s]

... (59 outputs left out)

Generating train split:   0%|          | 0/275512 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/275512 [00:00<?, ? examples/s]

Norwegian corpora collected: 6000 sentences.
Swedish corpora collected: 6000 sentences.
Danish corpora collected: 6000 sentences.
Non-Scandinavian corpora collected: 6000 sentences, with 1000.0 sentences per language.

Zero-Shot Classification Probabilities: 
Norwegian: 0.4851
Swedish: 0.4831
Danish: 0.4990
Non-Scandinavian: 0.5463


In [3]:
from datasets import Dataset, concatenate_datasets, Sequence, Value, Features
from sklearn.model_selection import train_test_split

def smoothed_values(target_labels, labels_list=labels, SMOOTH_VAL=0.15):
    if isinstance(target_labels, str):
        target_labels = [target_labels] # Accepts either a lang string or a lang list
    assert all(t in labels_list for t in target_labels), "Invalid target label(s)."
    
    num_labels = len(labels_list)
    num_targets = len(target_labels)
    new_list = [None] * num_labels

    for i, curr_label in enumerate(labels_list):
        is_target = curr_label in target_labels
        is_scandi = curr_label != "Non-Scandinavian"

        if is_target:
            new_list[i] = 1 - SMOOTH_VAL 
        elif is_scandi:
            no_da_pair = {"Norwegian", "Danish"}
            if no_da_pair.issubset(set(target_labels)):
                new_list[i] = SMOOTH_VAL / 3  # Swedish off-label
            else:
                da_no_cross = ("Norwegian" in target_labels and curr_label == "Danish") or ("Danish" in target_labels and curr_label == "Norwegian")
                new_list[i] = SMOOTH_VAL / (1.5 if da_no_cross else 3)
        else:
            new_list[i] = SMOOTH_VAL / 4  # Non-Scandinavian always low

    return new_list
    
norwegian_train = get_opus_sentences("en-nb", "nb") + get_opus_sentences("en-nn", "nn")
swedish_train = get_opus_sentences("en-sv", "sv")
danish_train = get_opus_sentences("da-en", "da")
nonscandi_train = []
for cfg, code in nonscandi_configs:
    nonscandi_train.extend(get_opus_sentences(cfg, code, sent_based_int(1 / len(nonscandi_configs))))

NUM_EVAL_SENTENCES = min(sent_based_int(0.15), 2000)

norwegian_eval = get_opus_sentences("en-nb", "nb", NUM_EVAL_SENTENCES, "validation") + get_opus_sentences("en-nn", "nn", NUM_EVAL_SENTENCES, "validation")
swedish_eval = get_opus_sentences("en-sv", "sv", NUM_EVAL_SENTENCES, "validation")
danish_eval = get_opus_sentences("da-en", "da", NUM_EVAL_SENTENCES, "validation")
nonscandi_eval = []
for cfg, code in nonscandi_configs:
    nonscandi_eval.extend(get_opus_sentences(cfg, code, NUM_EVAL_SENTENCES, "validation"))

# Very tiny makeshift corpus of sentences valid in multiple languages to avoid overconfidence in one lang.
all_scandi_sent = ["En dag i livet", "Vi har en bil.", "Han har en god plan.", "Herregud!", "Studenten ser en musiker.", "I går var det fint.",
                   "En normal person i en varm taxi kan se en film.", "Han har en bror i USA.", "De kan se gruppen.", "Min mormor eller min farmor?", "Solen går snart ned.", "Det var en kanin."]
da_no_cross_sent = ["Det kan jeg ikke.", "Dette er en fugl.", "Han spiste brødet.", "Hvem er du?", "Tyskland er et land.", "København er ikke en stor by.",
                    "Middagen er klar!", "Hun spiste et varmt måltid klokken 16.", "Dette er en norsk bygning.", "Er du sikker på det?", "Turen var for lang.", "Vi ses i morgen."]
all_scandi_labels = [smoothed_values(["Norwegian", "Swedish", "Danish"])] * len(all_scandi_sent)
da_no_cross_labels = [smoothed_values(["Norwegian", "Danish"])] * len(da_no_cross_sent)

CROSS_SCANDI_MULTIPLIER = max(1, int(sent_based_int(0.75) * 0.05 / (len(all_scandi_sent) + len(da_no_cross_sent))))

label_features = Features({"text": Value("string"), "labels": Sequence(Value("float32"))})

all_scandi_train, all_scandi_eval = train_test_split(
    list(zip(all_scandi_sent, all_scandi_labels)),
    test_size=0.15, random_state=42
)
da_no_cross_train, da_no_cross_eval = train_test_split(
    list(zip(da_no_cross_sent, da_no_cross_labels)),
    test_size=0.15, random_state=42
)
ms_train_texts, ms_train_labels = zip(*all_scandi_train)
ms_eval_texts, ms_eval_labels = zip(*all_scandi_eval)
da_no_cross_train_texts, da_no_cross_train_labels = zip(*da_no_cross_train)
da_no_cross_eval_texts, da_no_cross_eval_labels = zip(*da_no_cross_eval)

dataset_norwegian_train = Dataset.from_dict({"text": norwegian_train, "labels": [smoothed_values("Norwegian")] * len(norwegian_train)}, features=label_features)
dataset_swedish_train = Dataset.from_dict({"text": swedish_train, "labels": [smoothed_values("Swedish")] * len(swedish_train)}, features=label_features)
dataset_danish_train = Dataset.from_dict({"text": danish_train, "labels": [smoothed_values("Danish")] * len(danish_train)}, features=label_features)
dataset_nonscandi_train = Dataset.from_dict({"text": nonscandi_train, "labels": [smoothed_values("Non-Scandinavian")] * len(nonscandi_train)}, features=label_features)
dataset_multiscandi_train = Dataset.from_dict({"text": list(ms_train_texts) * CROSS_SCANDI_MULTIPLIER, "labels": list(ms_train_labels) * CROSS_SCANDI_MULTIPLIER}, features=label_features)
dataset_da_no_cross_train = Dataset.from_dict({"text": list(da_no_cross_train_texts) * CROSS_SCANDI_MULTIPLIER, "labels": list(da_no_cross_train_labels) * CROSS_SCANDI_MULTIPLIER}, features=label_features)
train_dataset = concatenate_datasets([dataset_norwegian_train, dataset_swedish_train, dataset_danish_train, dataset_nonscandi_train, dataset_multiscandi_train, dataset_da_no_cross_train]).shuffle(seed=42)

dataset_norwegian_eval = Dataset.from_dict({"text": norwegian_eval, "labels": [smoothed_values("Norwegian")] * len(norwegian_eval)}, features=label_features)
dataset_swedish_eval = Dataset.from_dict({"text": swedish_eval, "labels": [smoothed_values("Swedish")] * len(swedish_eval)}, features=label_features)
dataset_danish_eval = Dataset.from_dict({"text": danish_eval, "labels": [smoothed_values("Danish")] * len(danish_eval)}, features=label_features)
dataset_nonscandi_eval = Dataset.from_dict({"text": nonscandi_eval, "labels": [smoothed_values("Non-Scandinavian")] * len(nonscandi_eval)}, features=label_features)
dataset_multiscandi_eval = Dataset.from_dict({"text": list(ms_eval_texts) * CROSS_SCANDI_MULTIPLIER, "labels": list(ms_eval_labels) * CROSS_SCANDI_MULTIPLIER}, features=label_features)
dataset_da_no_cross_eval = Dataset.from_dict({"text": list(da_no_cross_eval_texts) * CROSS_SCANDI_MULTIPLIER, "labels": list(da_no_cross_eval_labels) * CROSS_SCANDI_MULTIPLIER}, features=label_features)
eval_dataset = concatenate_datasets([dataset_norwegian_eval, dataset_swedish_eval, dataset_danish_eval, dataset_nonscandi_eval, dataset_multiscandi_eval, dataset_da_no_cross_eval]).shuffle(seed=42)

print(train_dataset[:20]) # Test to see if dataset is selected + labeled properly

def tokenize_dataset(select_dataset):
    return tokenizer(select_dataset["text"], truncation=True, max_length=512)

print(f"{len(train_dataset)} Training samples | {len(eval_dataset)} Eval samples")

train_dataset = train_dataset.map(tokenize_dataset, batched=True)
eval_dataset = eval_dataset.map(tokenize_dataset, batched=True)
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

{'text': ['Sogar besser.', '- Ik ben nog niet klaar.', 'Tíkin ūín!', '_Fjern flagg', 'Omskipling', 'NOW WHAT ABOUT THAT CLOSED SIGN?', 'Parterne, der indgår specifikke delegeringsaftaler og aftaler om frivillig overdragelse af opgaver, skal nå til enighed om detaljerede vilkår, som mindst dækker følgende aspekter:', 'Kom dan, ik kwijn hier weg.', 'Endre den valgte kontoen', 'Benyttes til å konfigurere en bakgrunnsmotor', 'pounds (435g)', 'Tjena, knullfejs.', 'Kom tillbaka senare.', 'Viđ komumst kannski framhjá ūeim í myrkrinu.', 'varer, hvis naturlige farve efter ASTM D 1500 er mørkere end 3, og hvis viskositet ved 100 °C efter ASTM D 445 er 9 × 10- 6 m2 s- 1 og derover.', 'Lagrar kurvedata til ei fil.', "Article 1830 of the Civil Code states that a child must be under the age of 16 years at the time of the adoption petition or under 18 years at that time if before attaining the age of 16 the child has been, by law or de facto, under the adopter's care.", 'Årsrapport for 2008: Narkotik

Map:   0%|          | 0/30180 [00:00<?, ? examples/s]

Map:   0%|          | 0/9036 [00:00<?, ? examples/s]

In [4]:
import re
import numpy as np
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import f1_score

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_training, labels_list=labels): # Report each class' performance for trainng
    logits, soft_labels = eval_training
    probs = 1 / (1 + np.exp(-logits)) # Manual sigmoid
    preds = (probs > 0.5).astype(int)
    hard_labels = (soft_labels > 0.5).astype(int)

    f1_per_class = f1_score(hard_labels, preds, average=None, zero_division=0)
    macro_f1 = f1_score(hard_labels, preds, average="macro", zero_division=0)

    result = {"macro_f1": macro_f1}
    for name, f1 in zip(labels_list, f1_per_class):
        result[f"f1_{name.lower().replace('-', '_')}"] = f1
    return result

print("Phase 1: Freezing ScandiBERT \"tail\" for training...")
for param in model.roberta.parameters():
    param.requires_grad = False

training_args_phase1 = TrainingArguments(
    output_dir="./scandiprob_phase1",
    learning_rate=1e-3,                # Fast!      
    per_device_train_batch_size=8,
    num_train_epochs=2,                     
    weight_decay=0.01,
    fp16=False,
    report_to="none",
    dataloader_pin_memory=False,
    logging_steps=sent_based_int(0.1),
    eval_strategy="epoch"
)

trainer_phase1 = Trainer(
    model=model,
    args=training_args_phase1,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting Phase 1 Training...")
trainer_phase1.train()


print("\nPhase 2: Unfreezing for full fine-tuning...")
for param in model.roberta.parameters():
    param.requires_grad = True

training_args_phase2 = TrainingArguments(
    output_dir="./scandiprob_phase2",
    learning_rate=2e-5,                      # = 0.00002, "standard"
    per_device_train_batch_size=8,
    num_train_epochs=4,                      
    weight_decay=0.01,
    warmup_steps=sent_based_int(0.25),                    
    fp16=True,
    report_to="none",
    dataloader_pin_memory=False,
    logging_steps=sent_based_int(0.1),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1", # Ignore the bizarre output, this works
    greater_is_better=True
)

trainer_phase2 = Trainer(
    model=model,
    args=training_args_phase2,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting Phase 2 Training...")
trainer_phase2.train()
trainer_phase2.save_model("./scandiprob_full_raw")
print("Training complete and model saved!")

Phase 1: Freezing ScandiBERT "tail" for training...
Starting Phase 1 Training...


Epoch,Training Loss,Validation Loss,Macro F1,F1 Norwegian,F1 Swedish,F1 Danish,F1 Non Scandinavian
1,0.495551,0.437645,0.346676,0.630838,0.019417,0.002134,0.734313
2,0.475135,0.422939,0.532513,0.646782,0.686695,0.081883,0.714692


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (17 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete and model saved!


In [8]:
def nonscandi_penalty(text): # Regex scoring for negative class
    if not text.strip():
        return 1.0

    scandi_keyboard = r"[a-zA-ZæøåÆØÅäöÄÖéÉ0-9 !@#$%^&*()\-_=+\[\]{};':\",.<>?/`~\\|]"
    scandi_keyboard_alpha_only = r"[a-zA-ZæøåÆØÅäöÄÖéÉ ]"
    scandi_key_matches = re.findall(scandi_keyboard, text)
    scandi_alpha_matches = re.findall(scandi_keyboard_alpha_only, text)
    if len(scandi_alpha_matches) < (len(text) * 0.5):
        nonscandi_percent = 1.0
    else:
        nonscandi_percent = (1 - (len(scandi_key_matches) / len(text)))
    return nonscandi_percent

def da_no_cross_skew(text): # Linguistic heuristic scoring for differences between Norwegian and Danish
    if not text:
        return [0.0, 0.0]
    da_skew, no_skew = 0.0, 0.0
    text = text.strip().lower()
    # Danish Heuristics:
    # 1. "æ" followed by b, g, l, v, or t (vs. Norwegian that uses "e" usually)
    # 2. "ej" and "øj" (vs. Norwegian "ei" and "øy")
    # Norwegian Heuristics: 
    # 1. Double consanants at ends of words (vs. Danish that doesn't)
    # 2. "gj", "kj", "skj" spellings (vs. just Danish "g", "k", and "sk")
    da_no_regex = {r"æ[bgltv]": "DA", r"[eø]j" : "DA",
                   r"\b\w*([bdfgklnprst])\1\b" : "NO", r"(?:g|k|sk)j[eæø]" : "NO"}
    words = text.split()
    if not words:
        return [0.0, 0.0]
    skew_amount = 1.0 / len(text)
    
    for rule, lang in da_no_regex.items():
        rule_matches = len(re.findall(rule, text))
        skew_inc = rule_matches * skew_amount * (1.15 if len(text.split()) <= 6 else 1)
        if lang == "NO":
            no_skew += skew_inc
            da_skew -= skew_inc
        elif lang == "DA":
            da_skew += skew_inc
            no_skew -= skew_inc
    return [no_skew, da_skew]

def ScandiProb(text): # Actually showing the fine-tuned model some text to classify
    if not text.strip():
        return
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    model.eval()
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    print("\n")
    raw_probs = torch.sigmoid(outputs.logits)[0]

    nonscandi_ratio = nonscandi_penalty(text)
    no_skew, da_skew = da_no_cross_skew(text)
    
    final_probs = []
    skew_effect = False
    
    for i, label in enumerate(labels):
        prob = raw_probs[i].item()
        
        if label in ["Norwegian", "Swedish", "Danish"]:
            adjusted = prob * (1.0 - nonscandi_ratio)
        else: # Scale up/down by % of non-Scandi chars
            adjusted = prob + ((1.0 - prob) * nonscandi_ratio)

        if label == "Norwegian":
            adjusted = adjusted * (1.0 + no_skew)
            adjusted = adjusted * (1.0 - da_skew)
        elif label == "Danish":
            adjusted = adjusted * (1.0 + da_skew)
            adjusted = adjusted * (1.0 - no_skew)

        adjusted = min(1.0, max(0.0, adjusted))

        assert(0 <= adjusted <= 1)
            
        final_probs.append(adjusted)
    
        if (round(adjusted, 4) != round(prob, 4)):
            skew_effect = True

    if skew_effect:
        print("Pre-Adjustment Probabilities: ")
        for label, prob in zip(labels, raw_probs): # Print pre-regex-skew results!
            print(f"{label}: {prob:.4f}")
    print("\nPost-Adjustment Probabilities: " if skew_effect else "Post-Training Probabilities")
    for label, prob in zip(labels, final_probs): # Print post-regex-skew results!
        print(f"{label}: {prob:.4f}")

In [9]:
# Test cases, not using assert because I want to see resulting probs even if wrong

def scandiprob_test(input_text):
    print(input_text)
    ScandiProb(input_text)
    print("\n")

scandiprob_test(" ") # Testing regex for no chars at all
scandiprob_test(".")

scandiprob_test("Dette er en norsk setning.")
scandiprob_test("Detta är en svensk mening.")
scandiprob_test("Dette er en dansk sætning.")

scandiprob_test("Hetta er ein føroyskur setningur.") # Faroese, in pretraining but not in training
scandiprob_test("Dies ist ein deutscher Satz.") # German, in negative class training but not pretraining
scandiprob_test("Šis ir teikums latviešu valodā.") # Latvian, not in pretraining or training, but should be partially flagged by regex
scandiprob_test("これは日本語の文章です。") # Japanese, not in pretraining or training, but should be fully flagged by regex

scandiprob_test("En klar dag i dag") # Should be valid in Norwegian, Swedish, and Danish
scandiprob_test("Der stod ingen steder i Mai Buchs eksamenspapirer, at hun skulle havne pa et teater.") # Norwegian vs. Danish test, from SLIDE paper; Danish *should* prevail
scandiprob_test("Forfatteren skrev en roman om ungdomskriminalitet på en buss.") # A close-to-ambiguous Norwegian sentence.

# Excerpt from an Aftenposten (Norwegian) news article
scandiprob_test("Onsdag diskuterer EU hastetiltak som kan få ned forbruket av importert olje og gass. Ett av forslagene bringer frem minner fra pandemi-tiden: å oppfordre bedrifter til å innføre obligatorisk hjemmekontor en dag i uken, viser et lekket utkast sett av Financial Times og Politico.")
# Excerpt from a Dagens Nyheter (Swedish) news article
scandiprob_test("Det var en ansatt rikspolischef som klev ut genom den praktfulla dörren till justitieutskottets sammanträdessal i riksdagshuset. Petra Lundh var kallad för att svara på frågor om Polismyndighetens användning av AI-verktyg och eventuella affärer med den kontroversiella, amerikanska techjätten Palantir.")
# Excerpt from a Politiken (Danish) news article
scandiprob_test("Et loft over huslejen vil gøre det værre for lejerne. Det er den noget aparte påstand i et indlæg fra Christian Curdt-Christiansen (Pol 2.4.). Skribenten, der præsenterer sig selv som en 25-årig studerende, burde nok lige have deklareret, at han ifølge egne oplysninger er på lønningslisten hos lobbyorganisationen Bygherreforeningen [...]")

 


.


Pre-Adjustment Probabilities: 
Norwegian: 0.3801
Swedish: 0.1405
Danish: 0.2202
Non-Scandinavian: 0.0791

Post-Adjustment Probabilities: 
Norwegian: 0.0000
Swedish: 0.0000
Danish: 0.0000
Non-Scandinavian: 1.0000


Dette er en norsk setning.


Post-Training Probabilities
Norwegian: 0.8313
Swedish: 0.0477
Danish: 0.6333
Non-Scandinavian: 0.0052


Detta är en svensk mening.


Post-Training Probabilities
Norwegian: 0.0390
Swedish: 0.9222
Danish: 0.0390
Non-Scandinavian: 0.0253


Dette er en dansk sætning.


Pre-Adjustment Probabilities: 
Norwegian: 0.4476
Swedish: 0.0551
Danish: 0.8797
Non-Scandinavian: 0.0059

Post-Adjustment Probabilities: 
Norwegian: 0.4089
Swedish: 0.0551
Danish: 0.9592
Non-Scandinavian: 0.0059


Hetta er ein føroyskur setningur.


Post-Training Probabilities
Norwegian: 0.5067
Swedish: 0.0402
Danish: 0.0515
Non-Scandinavian: 0.3433


Dies ist ein deutscher Satz.


Post-Training Probabilities
Norwegian: 0.0339
Swedish: 0.0636
Danish: 0.0415
Non-Scandinavian: 0.8

In [10]:
# Different type of test case: calculating F1-scores post-regex skew to compare accuracy
import numpy as np
from sklearn.metrics import f1_score

def apply_post_skew(texts, raw_logits_array): # Returns post-skew probability array
    post_skew = []
    for text, logits in zip(texts, raw_logits_array):
        raw_probs = 1 / (1 + np.exp(-logits))
        nonscandi_ratio = nonscandi_penalty(text)
        no_skew, da_skew = da_no_cross_skew(text)
        adjusted = []
        for i, label in enumerate(labels):
            p = raw_probs[i]
            if label in ["Norwegian", "Swedish", "Danish"]:
                p = p * (1.0 - nonscandi_ratio)
            else:
                p = p + ((1.0 - p) * nonscandi_ratio)
            if label == "Norwegian":
                p = p * (1.0 + no_skew) * (1.0 - da_skew)
            elif label == "Danish":
                p = p * (1.0 + da_skew) * (1.0 - no_skew)
            adjusted.append(min(1.0, max(0.0, p)))
        post_skew.append(adjusted)
    return np.array(post_skew)

def print_f1(true_labels, preds, prefix=""):
    macro = f1_score(true_labels, preds, average="macro", zero_division=0)
    per_class = f1_score(true_labels, preds, average=None, zero_division=0)
    print(f"{prefix}macro F1: {macro:.6f}")
    for name, f1 in zip(labels, per_class):
        print(f"  {name}: {f1:.6f}")

norwegian_test = get_opus_sentences("en-nb", "nb", NUM_EVAL_SENTENCES, "test") + get_opus_sentences("en-nn", "nn", NUM_EVAL_SENTENCES, "test")
swedish_test = get_opus_sentences("en-sv", "sv", NUM_EVAL_SENTENCES, "test")
danish_test = get_opus_sentences("da-en", "da", NUM_EVAL_SENTENCES, "test")
nonscandi_test = []
for cfg, code in nonscandi_configs:
    nonscandi_test.extend(get_opus_sentences(cfg, code, NUM_EVAL_SENTENCES, "test"))

# Test datasets for final F1-scores
dataset_norwegian_test  = Dataset.from_dict({"text": norwegian_test, "labels": [smoothed_values("Norwegian")] * len(norwegian_test)}, features=label_features)
dataset_swedish_test    = Dataset.from_dict({"text": swedish_test, "labels": [smoothed_values("Swedish")] * len(swedish_test)}, features=label_features)
dataset_danish_test     = Dataset.from_dict({"text": danish_test, "labels": [smoothed_values("Danish")] * len(danish_test)}, features=label_features)
dataset_nonscandi_test  = Dataset.from_dict({"text": nonscandi_test, "labels": [smoothed_values("Non-Scandinavian")] * len(nonscandi_test)}, features=label_features)

test_dataset = concatenate_datasets([dataset_norwegian_test, dataset_swedish_test, dataset_danish_test, dataset_nonscandi_test]).shuffle(seed=42)

test_dataset = test_dataset.map(tokenize_dataset, batched=True)

test_texts = list(test_dataset["text"])
test_soft_labels = np.array(test_dataset["labels"]) 

test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

test_predictions = trainer_phase2.predict(test_dataset)
pre_hard_labels = (test_soft_labels > 0.5).astype(int)

print("Running ScandiProb post-skew against OPUS-100 test set...")

pre_preds = (1 / (1 + np.exp(-test_predictions.predictions)) > 0.5).astype(int)
print_f1(pre_hard_labels, pre_preds, "OPUS Test pre-skew ")

opus_preds = (apply_post_skew(test_texts, test_predictions.predictions) > 0.5).astype(int)
print_f1(pre_hard_labels, opus_preds, "OPUS Test post-skew ") # See if meaningful difference from regex skew

SLIDE_TO_SCANDIPROB = {
    "bokmål":  "Norwegian",
    "nynorsk": "Norwegian",
    "svensk":  "Swedish",
    "dansk":   "Danish",
    "annet":   "Non-Scandinavian",
}

def eval_match_labels(row):
    correct_langs = {row["choices"][i] for i in row["correct_indices"]}
    mapped = {SLIDE_TO_SCANDIPROB[lang] for lang in correct_langs if lang in SLIDE_TO_SCANDIPROB}
    return [1 if label in mapped else 0 for label in labels]

print("Running ScandiProb post-skew against SLIDE eval set...")

# Compare against SLIDE macro-F1 score of 96.6, with hybrid approach and data constraints in mind
slide_eval = load_dataset("ltg/slide", split="validation")
slide_texts = [row["text"] for row in slide_eval]
slide_true_labels = np.array([eval_match_labels(row) for row in slide_eval])

slide_raw_logits = []
for text in slide_texts:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    slide_raw_logits.append(outputs.logits[0].cpu().numpy())
slide_raw_logits = np.array(slide_raw_logits)

slide_preds = (apply_post_skew(slide_texts, slide_raw_logits) > 0.5).astype(int)
print_f1(slide_true_labels, slide_preds, "SLIDE test post-skew ")

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Running ScandiProb post-skew against OPUS-100 test set...
OPUS Test pre-skew macro F1: 0.930061
  Norwegian: 0.899132
  Swedish: 0.945290
  Danish: 0.916574
  Non-Scandinavian: 0.959249
OPUS Test post-skew macro F1: 0.930217
  Norwegian: 0.900299
  Swedish: 0.944695
  Danish: 0.916481
  Non-Scandinavian: 0.959392
Running ScandiProb post-skew against SLIDE eval set...


README.md: 0.00B [00:00, ?B/s]

train.jsonl:   0%|          | 0.00/52.5M [00:00<?, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/293802 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6433 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6950 [00:00<?, ? examples/s]

SLIDE test post-skew macro F1: 0.836058
  Norwegian: 0.884565
  Swedish: 0.841576
  Danish: 0.686937
  Non-Scandinavian: 0.931153


In [11]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_1 = user_secrets.get_secret("write-scandiprob")

login(token=secret_value_1)
repo_id = "ianro04/ScandiProb"

print(f"Pushing model and tokenizer to {repo_id}...")
model.push_to_hub(repo_id, private=True)
tokenizer.push_to_hub(repo_id, private=True)

print("Saved most recent iteration of ScandiProb.")

Pushing model and tokenizer to ianro04/ScandiProb...


README.md:   0%|          | 0.00/154 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/155 [00:00<?, ?B/s]

Saved most recent iteration of ScandiProb.
